# Kimi K2.7 Code — Cookbook

Practical recipes for building with Kimi K2.7 Code, an open-source coding-focused agentic model built for long-horizon software engineering.

> Source: Kimi K2.7 Code overview — https://www.kimi.com/resources/kimi-k2-7-code

Recommended model id:
- `kimi-k2.7-code`

Key facts from the docs:
- 256K context window
- thinking mode required
- text, image, and video input support
- optimized for long-horizon coding and agentic workflows

## 0) What Kimi K2.7 Code is good at

Kimi K2.7 Code is tuned for tasks that span many turns and many files.

Best fits:
- refactoring large codebases
- debugging across multiple modules
- agentic workflows with planning and verification
- code generation with strict implementation constraints
- multimodal coding tasks using text, image, and video input

Useful operating assumptions:
- keep the task scoped, but not tiny
- ask for a plan before edits
- require a validation step at the end
- use thinking mode for all requests

## 1) Setup

Install the packages you need for API access, structured outputs, and optional LangChain workflows.

- `pip install requests openai pydantic`
- Optional: `pip install langchain langchain-openai langchain-core langchain-text-splitters`

Environment variables to set:
- `KIMI_API_KEY`
- `KIMI_API_BASE` if your environment uses a custom base URL
- `KIMI_HTTP_REFERER` and `KIMI_APP_TITLE` for attribution where applicable

In [ ]:
import json
import os
from pathlib import Path

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
    print('Install the OpenAI SDK with: %pip install openai')

api_key = os.getenv('KIMI_API_KEY')
api_base = os.getenv('KIMI_API_BASE', 'https://api.moonshot.cn/v1')

if OpenAI is None:
    client = None
elif api_key:
    client = OpenAI(api_key=api_key, base_url=api_base)
    print('Kimi client configured for', api_base)
else:
    client = None
    print('Set KIMI_API_KEY to run live requests')

## 2) Thinking mode and basic request shape

Kimi K2.7 Code requires thinking mode. In practice, that means your request should always be written for a deliberate coding workflow: ask for a plan, implementation, and validation.

A good pattern is:
1. state the task
2. include constraints
3. ask for a short plan first
4. request the final artifact
5. ask for a validation step

In [ ]:
basic_request = {
    'model': 'kimi-k2.7-code',
    'messages': [
        {
            'role': 'system',
            'content': 'You are a senior coding agent. Always think carefully, produce a short plan, then implement.'
        },
        {
            'role': 'user',
            'content': 'Refactor the service layer to make database access testable. Return a plan, then the code changes, then validation steps.'
        }
    ],
    'temperature': 0.2,
    'max_tokens': 4096,
    'thinking': {'type': 'enabled'}
}

print(json.dumps(basic_request, indent=2))

## 3) Long-horizon coding recipe

This model is strongest when you give it a real engineering task with boundaries. Ask for a plan, file-level edits, and a validation pass.

Useful prompt ingredients:
- repo context or folder map
- explicit acceptance criteria
- files or modules that are in scope
- tests or checks that must pass

In [ ]:
long_horizon_prompt = '''
You are operating on a large codebase.
Task: implement a retry-safe background job runner for document sync.
Constraints:
- preserve existing public APIs
- add tests for success and retry failure paths
- keep changes minimal and readable
Output:
1. short plan
2. files to edit
3. implementation
4. tests
5. validation steps
'''

print(long_horizon_prompt)

## 4) Context caching and multi-turn workflows

Kimi API pricing includes automatic context caching, which is useful for repeated repository context, long specs, and multi-step debugging sessions.

Pattern:
- send stable project context once
- reuse the same conversation for follow-up edits
- keep repeated prompts compact
- ask for deltas instead of full rewrites when possible

In [ ]:
cached_context_prompt = {
    'system': 'You have already seen the repository context. Focus only on the requested delta.',
    'user': 'Update the auth flow to support short-lived tokens without changing the public login API.'
}

print(json.dumps(cached_context_prompt, indent=2))

## 5) LangChain integration

Use LangChain when you want prompt templates, chunking, retrievers, structured outputs, or a reusable agent workflow around Kimi K2.7 Code.

In [ ]:
# %pip install -U langchain langchain-openai langchain-core langchain-text-splitters

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

lc_model = ChatOpenAI(
    model='kimi-k2.7-code',
    api_key=os.getenv('KIMI_API_KEY'),
    base_url=api_base,
    temperature=0.2,
)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a senior coding agent. Think carefully, then answer with practical engineering detail.'),
    ('user', 'Create a short plan and implementation outline for: {task}'),
])

chain = prompt | lc_model | StrOutputParser()
print('LangChain chain prepared for Kimi K2.7 Code')

## 6) Multimodal coding workflows

Kimi K2.7 Code supports text, image, and video input. That makes it useful for tasks like UI debugging, visual diff inspection, and code-understood-from-screenshot workflows.

Ask for:
- a plain-language diagnosis
- code-level implications
- a concise action list
- any assumptions the model had to make

In [ ]:
multimodal_prompt = {
    'system': 'Analyze the image or video frame as a coding assistant. Be concrete and avoid guessing.',
    'user': 'Look at this UI screenshot and identify the most likely cause of the layout bug, then suggest the smallest safe code fix.'
}

print(json.dumps(multimodal_prompt, indent=2))

## 7) Structured outputs and evaluation

For agentic coding tasks, schema-based outputs make it easier to route work between planning, implementation, and review steps.

Good schema examples:
- task summary
- milestones
- risks
- validation checks

In [ ]:
from pydantic import BaseModel, Field
from typing import List

class KimiActionPlan(BaseModel):
    summary: str = Field(..., description='Short task summary')
    milestones: List[str] = Field(..., description='Ordered implementation milestones')
    risks: List[str] = Field(..., description='Key technical risks')
    validation: List[str] = Field(..., description='Checks to run after implementation')

structured_prompt = {
    'system': 'Return a structured action plan for a long-horizon coding task.',
    'user': 'Plan a migration of a monolith auth flow into a token-based service boundary.'
}

print('Structured schema ready:', KimiActionPlan.model_json_schema()['title'])
print(json.dumps(structured_prompt, indent=2))

## 8) Practical prompt checklist

Before sending a task to Kimi K2.7 Code, check:
- Is the task scoped enough to finish?
- Did you include repo context or examples?
- Did you ask for a plan and validation?
- Are you using thinking mode?
- Is the requested output format explicit?

In [ ]:
def choose_model(task_kind: str) -> str:
    if task_kind in {'deep_refactor', 'multi_file_debugging', 'agentic_repo_work'}:
        return 'kimi-k2.7-code'
    if task_kind in {'quick_classification', 'short_summary'}:
        return 'kimi-k2.6'
    return 'kimi-k2.7-code'

for task_kind in ['deep_refactor', 'quick_classification', 'agentic_repo_work']:
    print(task_kind, '->', choose_model(task_kind))